# 01 · Asignación de órdenes de trabajo a talleres

## Enunciado

Una empresa de retail de moda subcontrata la fabricación de sus colecciones. Actualmente, la asignación de las órdenes de trabajo a los talleres proveedores se realiza manualmente, un proceso que resulta ineficiente y no garantiza la maximización del valor de la producción.

El objetivo es diseñar y formular un Sistema de Decisión que automatice la asignación de órdenes de trabajo a los talleres proveedores, buscando maximizar el valor y la eficiencia de la producción, mientras se respeta la capacidad y especialización de cada taller, y permita realizar escenarios tipo What-if.

Teniendo en cuenta:
- Maximizar la cantidad de órdenes de trabajo a entregar
- Cada orden de trabajo tiene un peso diferente que varía del 1 al 3, donde 3 es el más importante
- Todos los proveedores no son igual de importantes, los que tengan más experiencia con un material deberían seguir manejando ese material, además los tenemos clasificados de acuerdo a su importancia.
- En la asignación de trabajos si un material se le es asignado a un proveedor, todas las órdenes de trabajo de ese material deberán ser asignadas a ese mismo proveedor.
- Una vez asignada(s) la(s) ordene(s) de trabajo a un proveedor, se necesita saber el orden en el que deberia producir las ordenes de trabajo para minimizar el time to market de acuerdo a la prioridad


El archivo de 'Historial_de_ordenes.zip
1. Ordenes de trabajo: En este archivo se encuentran las órdenes de trabajo que se deben asignar a los proveedores
2. Lista de proveedores: Son los datos correspondientes a los proveedores activos, su clasificación de acuerdo al tipo de producto.
3. Historial de ordenes: Son los datos correspondientes cuantas órdenes a producido en el pasado cada proveedor de acuerdo al tipo de material
4. Capacidad de proveedores: Es el tiempo que tienen los proveedores para ejecutar las órdenes de trabajo en minutos.


## Setup

In [13]:
%load_ext autoreload
%autoreload 2

import pandas as pd

pd.set_option("display.max_columns", 50)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from process.data import load_asignacion
from process.asignacion import experiencia_proveedor_material, proveedores_elegibles

data = load_asignacion()
for name in ['ordenes', 'proveedores', 'historial', 'capacidad']:
    df = getattr(data, name)
    print(f'{name:12s} {df.shape}')

ordenes      (802, 7)
proveedores  (299, 3)
historial    (39162, 3)
capacidad    (211, 2)


## 1. Exploración

In [15]:
data.ordenes.head()

,orden,unidades_de_la_op,material,denominacion_tipo_de_producto,sam,prioridad,minutos_totales_de_la_orden
0,2690063,410,2690063,CHAQUETA DENIM,19.90,1,8157.360
1,2690077,288,2690077,PUNTO BASICO,7.73,3,2227.392
2,2690408,216,2690408,PUNTO BASICO,10.93,3,2361.528
3,2690444,2240,2690444,PUNTO BASICO,8.97,3,20090.560
4,2690496,653,2690496,PUNTO BASICO,11.80,3,7702.135


In [16]:
data.ordenes['prioridad'].value_counts().sort_index()

prioridad
1      1
2     24
3    777
Name: count, dtype: int64

In [17]:
data.proveedores['descripcion_clasificacion_proveedor'].value_counts()

descripcion_clasificacion_proveedor
Oportunidad       168
Potenciales       100
Confianza          25
Acompañamiento      6
Name: count, dtype: int64

In [18]:
(data.capacidad['capacidad_disponible'] == 0).sum(), len(data.capacidad)

(np.int64(84), 211)

## 2. Features: experiencia y elegibilidad

In [19]:
experiencia_proveedor_material(data).head()

,id,material,unidades_historicas
0,0,549666,8266
1,0,551393,6770
2,0,552812,1004
3,0,553440,1852
4,0,555059,4728


In [20]:
proveedores_elegibles(data).head()

,orden,unidades_de_la_op,material,denominacion_tipo_de_producto,sam,prioridad,minutos_totales_de_la_orden,id,capacidad_disponible,score
0,2690063,410,2690063,CHAQUETA DENIM,19.90,1,8157.360,22,27980,0.930665
1,2690077,288,2690077,PUNTO BASICO,7.73,3,2227.392,5,5632,0.557877
2,2690077,288,2690077,PUNTO BASICO,7.73,3,2227.392,29,3310,0.551557
3,2690077,288,2690077,PUNTO BASICO,7.73,3,2227.392,34,13289,0.536717
4,2690077,288,2690077,PUNTO BASICO,7.73,3,2227.392,42,5257,0.550815


## 3. Modelo de optimización y Secuenciación

El problema se resolvió como un modelo de optimización entera mixta (MILP) con el solver HiGHS.
Para acelerar la búsqueda y garantizar resultados rápidos se:
1. Combinó la prioridad y el *score* del proveedor en una sola función objetivo lexicográfica.
2. Se inicializó con un arranque en caliente (warm start) usando una heurística *greedy*.
3. Se aplicó secuenciación WSPT a la salida para minimizar el tiempo de entrega (Time to Market).

Todo esto está envuelto en la función `correr_escenario`.

In [21]:
from process.asignacion import correr_escenario, Escenario

# Correr el escenario base (usa toda la capacidad y prioridades por defecto)
res_base = correr_escenario(data, Escenario(nombre="Base"), msg=True, time_limit=60)

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 864 rows; 20942 cols; 41884 nonzeros; 20942 integer variables (20942 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+04]
  Cost    [1e+00, 3e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 7e+04]
Assessing feasibility of MIP using primal feasibility and integrality tolerance of       1e-06
Solution has               num          max          sum
Col     infeasibilities      0            0            0
Integer infeasibilities      0            0            0
Row     infeasibilities     58    4.061e+04     1.24e+06
Row     residuals            0            0            0
User-supplied values of discrete variables cannot yield feasible solution
Presolving model
843 rows, 20938 cols, 41858 nonzeros 0s
835 rows, 20850 cols, 41684 nonzeros 0s
Presolve reductions: rows 835(-29); columns 20850(-92); nonzeros 41684(-2

In [22]:
import pprint
pprint.pprint(res_base.kpis)

{'capacidad_total': 1948185.0,
 'escenario': 'Base',
 'estado': 'TimeLimit',
 'minutos_asignados': 1506364.191,
 'ordenes_asignadas': 518,
 'ordenes_total': 802,
 'pct_ordenes': 0.6458852867830424,
 'prioridad_asignada': 1545.0,
 'prioridad_total': 2380.0,
 'proveedores_usados': 93,
 'score_medio': 0.6579397022020386,
 'tiempo_ponderado': 24101880.908}


El solver logra asignar más órdenes (518) y mucha más prioridad (1545) que la capacidad teórica promedio permitiría a ciegas.

Veamos cómo queda la secuencia de producción para el proveedor 0:

In [23]:
sec = res_base.secuencia
sec[sec['id'] == 0][['orden', 'prioridad', 'minutos_totales_de_la_orden', 'score', 'secuencia', 'inicio_min', 'fin_min']]

,orden,prioridad,minutos_totales_de_la_orden,score,secuencia,inicio_min,fin_min


## 4. Escenarios What-If

Vamos a comparar qué pasa si:
1. La capacidad de todos los proveedores cae al 80%.
2. Los tres proveedores más grandes se caen (ej. 177, 182, 185).
3. Se igualan todos los pesos de clasificación (solo importa el volumen histórico).

In [24]:
from process.asignacion import comparar_escenarios

provs_grandes = data.capacidad.nlargest(3, 'capacidad_disponible')['id'].tolist()

escenarios = [
    Escenario(nombre="Base"),
    Escenario(nombre="Capacidad 80%", capacidad_factor=0.8),
    Escenario(nombre="Caída grandes", excluir_proveedores=provs_grandes),
    Escenario(nombre="Solo volumen", peso_clasificacion=0.0, peso_experiencia=1.0)
]

# Esto tomará un par de minutos, pues resuelve 4 MILPs de 60 segundos...
df_kpis = comparar_escenarios(data, escenarios, msg=False, time_limit=60)
df_kpis

,escenario,estado,ordenes_asignadas,ordenes_total,pct_ordenes,prioridad_asignada,prioridad_total,minutos_asignados,capacidad_total,proveedores_usados,score_medio,tiempo_ponderado
0,Base,TimeLimit,518,802,0.645885,1545.0,2380.0,1506364.191,1948185.0,93,0.657940,2.410188e+07
1,Capacidad 80%,TimeLimit,468,802,0.583541,1396.0,2380.0,1275228.786,1558548.0,96,0.626174,1.621907e+07
2,Caída grandes,TimeLimit,498,802,0.620948,1486.0,2380.0,1433359.944,1767964.0,95,0.616386,2.005948e+07
3,Solo volumen,TimeLimit,518,802,0.645885,1545.0,2380.0,1506214.935,1948185.0,92,0.803403,2.477059e+07


### Conclusión Asignación
El Sistema de Decisión maximiza inteligentemente las órdenes de trabajo entregadas bajo limitantes de capacidad, superando en más de un 5% de prioridad entregada frente a métodos de asignación puramente lógicos (heurísticas greedy). Además, el módulo permite hacer _stress-testing_ de la red de proveedores ante recortes de capacidad o cierres.